In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
emp_data = [(1,'manish',50000,'IT','m'),
(2,'vikash',60000,'sales','m'),
(3,'raushan',70000,'marketing','m'),
(4,'mukesh',80000,'IT','m'),
(5,'priti',90000,'sales','f'),
(6,'nikita',45000,'marketing','f'),
(7,'ragini',55000,'marketing','f'),
(8,'rashi',100000,'IT','f'),
(9,'aditya',65000,'IT','m'),
(10,'rahul',50000,'marketing','m'),
(11,'rakhi',50000,'IT','f'),
(12,'akhilesh',90000,'sales','m')]

emp_df = spark.createDataFrame(emp_data,['id','name','salary','dept','gender'])
emp_df.createOrReplaceTempView("emp")


In [0]:
win = Window.partitionBy('dept')
emp_df = emp_df.withColumn("department_cost",sum("salary").over(win))\
    .withColumn("emp_count",count("id").over(win))\
    # .withColumn('cost_percentage',expr("(salary/department_cost)*100").cast("decimal(5,2)"))
emp_df.show(truncate=False)

In [0]:
winrank = Window.partitionBy('dept','gender').orderBy(col('salary').asc())
emp_dfrow_num = emp_df.withColumn("row_num",row_number().over(winrank))\
    .withColumn("dense_rank",dense_rank().over(winrank))\
    .withColumn("rank",rank().over(winrank))\
    .filter(col('rank')==1)

emp_dfrow_num.show(truncate=False)

In [0]:
product_data = [
(1,"iphone","01-01-2023",1500000),
(2,"samsung","01-01-2023",1100000),
(3,"oneplus","01-01-2023",1100000),
(1,"iphone","01-02-2023",1300000),
(2,"samsung","01-02-2023",1120000),
(3,"oneplus","01-02-2023",1120000),
(1,"iphone","01-03-2023",1600000),
(2,"samsung","01-03-2023",1080000),
(3,"oneplus","01-03-2023",1160000),
(1,"iphone","01-04-2023",1700000),
(2,"samsung","01-04-2023",1800000),
(3,"oneplus","01-04-2023",1170000),
(1,"iphone","01-05-2023",1200000),
(2,"samsung","01-05-2023",980000),
(3,"oneplus","01-05-2023",1175000),
(1,"iphone","01-06-2023",1100000),
(2,"samsung","01-06-2023",1100000),
(3,"oneplus","01-06-2023",1200000)
]
product_df = spark.createDataFrame(product_data,['id','name','date_of_purchase','price'])
product_df.createOrReplaceTempView("product")


In [0]:
product_df.display()

# Lead and lag

In [0]:
sales_win = Window.partitionBy("name").orderBy("date_of_purchase")
sales_df = product_df.withColumn("last_price",lag(col("price"),1).over(sales_win))
sales_df =  sales_df.withColumn("earn_percentage",expr("(price-last_price)/last_price*100"))
sales_df.display()

In [0]:
sales_df.select(col("earn_percentage").cast("decimal(5,2)").alias("decimal_per"),round(col("earn_percentage"),2).alias("round_per"),bround(col("earn_percentage"),2).alias("bround_per"),
                floor(col("earn_percentage")),ceil(col("earn_percentage")) ).display()


In [0]:
last_sales_win = Window.partitionBy("id").orderBy("date_of_purchase").rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)
# last_sales_df1 = sales_df.withColumn("total_price",sum(col("price")).over(last_sales_win))\
#     .withColumn("total_earn_percentage",bround((col("price")/col("total_price"))*100, 2)).explain()

last_df = sales_df.withColumn('first_sale')

# last_sales_df1.display()

In [0]:
weather_data = [
    (794, "Kissee Mills", "MO", 140, 73),
    (824, "Loma Mar", "CA", 49, 131),
    (603, "Sandy Hook", "CT", 72, 148),
    (478, "Tipton", "IN", 34, 98),
    (619, "Arlington", "CO", 75, 93),
    (711, "Turner", "AR", 50, 101),
    (839, "Slidell", "LA", 85, 152),
    (411, "Negreet", "LA", 99, 105),
    (588, "Glencoe", "KY", 46, 136),
    (665, "Chelsea", "IA", 99, 60),
    (342, "Chignik Lagoon", "AK", 103, 153),
    (733, "Pelahatchie", "MS", 39, 28),
    (441, "Hanna City", "IL", 51, 137),
    (811, "Dorrance", "KS", 102, 122),
    (698, "Albany", "CA", 50, 80),
    (325, "Monument", "KS", 71, 142),
    (414, "Manchester", "MD", 74, 37),
    (113, "Prescott", "IA", 40, 66),
    (971, "Graettinger", "IA", 95, 150),
    (266, "Cahone", "CO", 116, 127),
    (617, "Sturgis", "MS", 36, 126),
    (495, "Upperco", "MD", 114, 30),
    (473, "Highwood", "IL", 27, 151),
    (959, "Waipahu", "HI", 106, 34),
    (438, "Bowdon", "GA", 89, 78)
]

weather_df = spark.createDataFrame(weather_data,['id','city','state','long_w','lat_w'])
weather_df.createOrReplaceTempView("weather")
weather_df.display()

In [0]:
weather_df1 = weather_df.select(col("city")).distinct().where(col("id")%2 == 0)
weather_df1.display()

In [0]:
weather_df.select(count(col('city'))-countDistinct(col("city"))).display()

In [0]:
weather_df.select("city").distinct().filter(~col('city').like('A%')).display()

In [0]:
weather_df.select('city').dropDuplicates().filter(~col('city').like('A%')).display()

In [0]:
# weather_df.filter(col("city").rlike('[aeiou]$')).show() # ends with vowel
# weather_df.filter(col("city").rlike("^[AEIOU]")).show() # starts with vowels


In [0]:

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("marks", IntegerType(), True)
])

In [0]:
data = [
    (19, "Samantha", 87),
    (21, "Julia", 96),
    (11, "Britney", 95),
    (32, "Kristeen", 100),
    (12, "Dyana", 55),
    (13, "Jenny", 66),
    (14, "Christene", 88),
    (15, "Meera", 24),
    (16, "Priya", 76),
    (17, "Priyanka", 77),
    (18, "Paige", 74),
    (19, "Jane", 64),
    (21, "Belvet", 78),
    (31, "Scarlet", 80),
    (41, "Salma", 81),
    (51, "Amanda", 34),
    (61, "Heraldo", 94),
    (71, "Stuart", 99),
    (81, "Aamina", 77),
    (76, "Amina", 89),
    (91, "Vivek", 84),
    (17, "Evil", 79),
    (16, "Devil", 76),
    (34, "Fanny", 75),
    (38, "Danny", 75)
]
students_df = spark.createDataFrame(data, schema)



In [0]:
students_df.select(col("name"))\
.filter(col("marks") > 75)\
.orderBy(right(col("name"), lit(3)),col("id")).display()

#pyspark string functions

## case conversion

In [0]:
students_df.select(upper(col("name")).alias("name_upper"),
                   lower(col("name")).alias("name_lower"),
                   initcap(col("name")).alias("name_init")
                   
                   ).display()

## Length

In [0]:
students_df.select(length("name")).show()

## substring

In [0]:
students_df.select(substring("name",3,3)).show()

In [0]:
students_df.select(concat(col("id"),lit("-"),col("name")).alias("unique_key")).display()

## Trim spaces

In [0]:
students_df.select(trim('name')).show()

In [0]:
students_df.select(regexp_replace("name","a","@")).display()

In [0]:
students_df.select(split("name","a").getItem(0)).show()


In [0]:
# students_df.filter(col("name").contains("n")).show()
students_df.filter(col("name").like("%jo%")).show()

In [0]:
students_df.select(reverse("name")).show()

In [0]:
students_df.filter(col("marks")>75).select(count("name")).show()
students_df.fitler(col("marks")>75).select(countDistinct("name"))

In [0]:
data1 = [(1,'A','2024-01-01'),
         (1,'A','2024-02-01'),
         (2,'B','2024-01-01'),
         (3,'C','2024-02-01'),
         (4,'C','2024-04-01')
         ]
duplicate_df = spark.createDataFrame(data1,["id","name","date"])

In [0]:
win_dup = Window.partitionBy("name").orderBy(col('date').desc())
duplicate_df.withColumn("row_no",row_number().over(win_dup))\
    .filter(col("row_no")==1).show()